# Classify a future low-SNR condition
**Objective:** construct a training-defined event, fit probability predictions, select a decision threshold and diagnose prevalence shift.
The event is relative low SNR, not a delivery failure or a verified incident.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


In [ ]:
case=MATCHED/'uva_sensor03_snr_h6';a=np.load(case/'examples.npz',allow_pickle=False)
X=a['X'];snr=a['y'][:,0]
tr,va,te=[np.flatnonzero(a['split']==s) for s in ['train','validation','test']]
q=np.quantile(snr[tr],.2);y=(snr<q).astype(int)
print('Fixed target threshold dB:',q)
display(pd.Series({part:y[idx].mean() for part,idx in [('train',tr),('validation',va),('test',te)]},name='event prevalence'))


## 1. Fit probability models and select by validation log loss

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss,brier_score_loss,average_precision_score,f1_score,confusion_matrix
choices=[]
for C in [.01,.1,1,10]:
    fitted=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=2000)).fit(X[tr],y[tr])
    choices.append((log_loss(y[va],fitted.predict_proba(X[va])[:,1]),C,fitted))
_,C,model=min(choices,key=lambda z:z[0])
pva=model.predict_proba(X[va])[:,1];ptest=model.predict_proba(X[te])[:,1]
threshold=max(np.arange(.05,1,.05),key=lambda t:f1_score(y[va],pva>=t,zero_division=0))
print('Selected C:',C,'decision threshold:',threshold)
display(pd.Series({'Brier':brier_score_loss(y[te],ptest),'AP':average_precision_score(y[te],ptest),'F1':f1_score(y[te],ptest>=threshold)}))
display(pd.DataFrame(confusion_matrix(y[te],ptest>=threshold),index=['actual 0','actual 1'],columns=['predicted 0','predicted 1']))


## 2. Inspect calibration and counts, not only a ranking metric

In [ ]:
bins=np.minimum((ptest*5).astype(int),4)
reliability=pd.DataFrame({'bin':bins,'p':ptest,'y':y[te]}).groupby('bin').agg(mean_probability=('p','mean'),event_fraction=('y','mean'),n=('y','size'))
display(reliability)
fig,ax=plt.subplots(figsize=(4,4));ax.plot([0,1],[0,1],linestyle=':',color='black');ax.plot(reliability.mean_probability,reliability.event_fraction,'o-');ax.set(xlabel='Predicted probability',ylabel='Observed fraction');plt.show()
archived=pd.read_csv(GUIDE/'metrics.csv')
display(archived[archived.task.eq('classification')].groupby(['case','model'])[['brier','ap','f1']].mean())


**Exercise:** why is test prevalence not necessarily 20%? Does a high per-hour F1 prove early incident warning?

<details><summary>Feedback after your attempt</summary>

the threshold was fixed from training; the future distribution can change. Several correctly classified hours in an ongoing episode do not establish onset anticipation. Episode definitions, lead time and alarm burden require another evaluation.

</details>

**Adaptation task:** define your actual event and threshold before using the test. Compare probability quality and an operational decision criterion separately.